In [1]:
%load_ext autoreload
%autoreload 2

import pathlib
import sys
project_root = pathlib.Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import ISRUtilities as isru

import numpy as np
import fmatoolbox as fma
import torch
import sklearn.model_selection

froot = pathlib.Path().cwd().parent.parent / 'Results/Figures/ISIntervals'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
do_save = False

In [2]:
import torch.nn as nn

class CNN1D(nn.Module):
    def __init__(self, n_features, n_time):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=n_features,out_channels=32,kernel_size=7,padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=32,out_channels=64,kernel_size=7,padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Linear(64,1)

    def forward(self,x):
        x = self.conv(x)
        x = x.squeeze(-1)
        return self.fc(x).squeeze(-1)

In [3]:
def _couplingCNN(session):

    events = isru.loadHpcPfcEvents(session)

    # load data around ripples
    data, t = fma.data.loadWideband(session,channels=np.arange(100,120),intervals=events['ripples'][:,None]-[0.3,0.05],cat=False)
    n_samples = min([len(d) for d in data])
    data = np.stack([d[:n_samples].T for d in data],axis=0) # (events, channels, time)
    t = np.stack([d[:n_samples] for d in t],axis=0) # (events, time)

    # smallest delta time following each event ripple
    idx = np.searchsorted(events['deltaWaves'],events['ripples'],side='right')
    valid = idx < len(events['deltaWaves'])
    distance = np.full_like(events['ripples'],np.nan,dtype=float)
    distance[valid] = events['deltaWaves'][idx[valid]] - events['ripples'][valid]
    valid = ~np.isnan(distance)
    data = data[valid]
    distance = distance[valid]

    X = torch.tensor(data,dtype=torch.float32)
    y = torch.tensor(distance,dtype=torch.float32)

    kf = sklearn.model_selection.KFold(n_splits=5,shuffle=True,random_state=42)
    fold_results = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f"fold {fold+1}")

        # split into train and validation sets
        X_train = X[train_idx]
        y_train = y[train_idx]
        X_val = X[val_idx]
        y_val = y[val_idx]

        # normalize
        m = X_train.mean(dim=(0,2),keepdim=True)
        s = X_train.std(dim=(0,2),keepdim=True)
        X_train = (X_train - m) / s
        X_val = (X_val - m) / s

        # repeat data for 50 epochs
        train_ds = torch.utils.data.TensorDataset(X_train,y_train)
        val_ds = torch.utils.data.TensorDataset(X_val,y_val)
        train_loader = torch.utils.data.DataLoader(train_ds,batch_size=16,shuffle=True)
        val_loader = torch.utils.data.DataLoader(val_ds,batch_size=16,shuffle=False)

        model = CNN1D(n_features=X.shape[1],n_time=X.shape[2])
        optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
        criterion = torch.nn.MSELoss()

        # training
        for epoch in range(50):
            model.train()
            for xb, yb in train_loader:
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred,yb)
                loss.backward()
                optimizer.step()

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                loss = criterion(pred,yb)
                val_losses.append(loss.item())
        mean_val_loss = np.mean(val_losses)
        print(f"Validation MSE: {mean_val_loss:.4f}")
        fold_results.append(mean_val_loss)

    return fold_results

In [4]:
session = fma.data.readBatchFile(batch_file)[0][4]
print(session)
_couplingCNN(session)

/mnt/hubel-data-131/perceval/Rat003_20231218/Rat003_20231218.xml
fold 1


Error removing breakpoint: Breakpoint id not found: fbb137d63c9eced11ee2f27402818e7a0fbda8b7fe57fe346b52218b86b90d6b id: 70. Available ids: [72, 12]

Error removing breakpoint: Breakpoint id not found: fbb137d63c9eced11ee2f27402818e7a0fbda8b7fe57fe346b52218b86b90d6b id: 12. Available ids: [70, 72]



Validation MSE: 878589.9437
fold 2
Validation MSE: 112377.8683
fold 3
Validation MSE: 520282.3176
fold 4
Validation MSE: 60348.4700
fold 5
Validation MSE: 48577.3151


[np.float64(878589.9437477805),
 np.float64(112377.86834161932),
 np.float64(520282.3176269531),
 np.float64(60348.46999289773),
 np.float64(48577.315118963066)]

In [6]:
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)
print(torch.cuda.is_available())

13.0
True
